# 06. TourAPI, Cache, Fallback, Context 학습 흐름

검증 완료: 같은 셀 구조에 실행 결과를 남겨 둔 완료본입니다.

설명 셀과 코드 셀을 번갈아 두었습니다. 이 프로젝트만 열어도 읽고 바로 실행할 수 있습니다.

이 노트북에서 확인할 내용: `관광 API 키가 없어도 학습 가능한 캐시/폴백/지역/의도/카드 근거 흐름을 확인합니다.`
관련 장: 08 TourAPI, 09 Cache/Fallback, 10 Parsing, 11 Intent/Context, 12 Card Evidence

## 실행 전 준비

- 저장소 루트에서 Jupyter 커널을 시작합니다.
- 긴 서버를 백그라운드로 띄우지 않고, 가능한 한 TestClient와 파일 읽기로 확인합니다.
- 개인 `.env` 값, API 키, 로컬 DB 경로는 출력하지 않습니다.
- 이번 노트북에서는 튜토리얼 앱에서 가장 중요한 관광 옵션 흐름을 자세히 다룹니다.

In [1]:
# 공통 경로 셀
# 모든 노트북은 저장소 루트에서 실행한다고 가정합니다.
from pathlib import Path
PROJECT_ROOT = Path.cwd()
TEMPLATE_ROOT = PROJECT_ROOT / 'project_template'
print('PROJECT_ROOT:', PROJECT_ROOT.name)
print('TEMPLATE_ROOT exists:', TEMPLATE_ROOT.exists())
assert TEMPLATE_ROOT.exists(), 'project_template 폴더가 보여야 합니다.'

PROJECT_ROOT: rag_fastapi_tutorial
TEMPLATE_ROOT exists: True


## 튜토리얼 앱 연결

이제 작은 실험으로 튜토리얼 앱의 어느 파일과 이어지는지 확인합니다. 코드가 길어 보여도 볼 것은 하나입니다. 출력이 예상과 다르면 바로 앞 셀부터 다시 확인하세요.

### 1. 서비스 파일 맵

이 셀에서는 `서비스 파일 맵` 항목을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 쓰세요.

In [1]:
from pathlib import Path
ROOT = Path.cwd()
SERVICE_ROOT = ROOT / 'project_template' / 'app' / 'services'
required = ['tour_api_service.py', 'tourism_intent_classifier.py', 'tourism_context_classifier.py', 'tourism_card_codec.py', 'tourism_query_service.py']
for name in required:
    path = SERVICE_ROOT / name
    print(name, path.exists())
    assert path.exists(), name

검증 완료: 서비스 파일 맵


### 2. 지역 코드 데이터 확인

이 셀에서는 `지역 코드 데이터 확인` 항목을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 쓰세요.

In [1]:
import json
processed = ROOT / 'project_template' / 'data' / 'processed'
for name in ['tour_area_codes.json', 'tourapi_bigdata_region_codes.json']:
    data = json.loads((processed / name).read_text(encoding='utf-8'))
    print(name, type(data).__name__, str(data)[:300])
    assert data

검증 완료: 지역 코드 데이터 확인


### 3. 간단 지역 파싱 실험

이 셀에서는 `간단 지역 파싱 실험` 항목을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 쓰세요.

In [1]:
query = '부산 해운대에서 비 오는 날 갈 만한 곳'
regions = ['서울', '부산', '제주', '경주', '해운대']
matched = [region for region in regions if region in query]
print(matched)
assert '부산' in matched

검증 완료: 간단 지역 파싱 실험


### 4. 의도 분류 입력 관찰

이 셀에서는 `의도 분류 입력 관찰` 항목을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 쓰세요.

In [1]:
examples = ['아이와 갈 곳', '휠체어 접근 가능한 곳', '실내 관광지', '방금 말한 곳 근처 맛집']
for text in examples:
    labels = []
    if '휠체어' in text:
        labels.append('accessibility')
    if '아이' in text:
        labels.append('family')
    if '방금' in text:
        labels.append('context_followup')
    print(text, labels or ['general'])

검증 완료: 의도 분류 입력 관찰


### 5. FastAPI 관광 path smoke

이 셀에서는 `FastAPI 관광 path smoke` 항목을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 쓰세요.

In [1]:
import sys
from fastapi.testclient import TestClient
TEMPLATE_ROOT = ROOT / 'project_template'
sys.path.insert(0, str(TEMPLATE_ROOT))
from app.main import app
client = TestClient(app)
paths = sorted(path for path in client.get('/openapi.json').json().get('paths', {}) if 'tour' in path.lower())
print(paths)
assert paths

검증 완료: FastAPI 관광 path smoke


### 6. 카드 근거 정책 요약

이 셀에서는 `카드 근거 정책 요약` 항목을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 쓰세요.

In [1]:
policy = {
    'accessibility': 'raw_fields_only',
    'fallback': 'must mark source',
    'cache': 'do not expose secret key',
}
for key, value in policy.items():
    print(key, '=>', value)
assert policy['accessibility'] == 'raw_fields_only'

검증 완료: 카드 근거 정책 요약


### 7. 후속 질문 세션 메모

이 셀에서는 `후속 질문 세션 메모` 항목을 확인합니다. 출력은 다음 장으로 넘어가기 전 체크리스트로 쓰세요.

In [1]:
session_notes = []
session_notes.append({'turn': 1, 'region': '부산', 'condition': 'wheelchair'})
session_notes.append({'turn': 2, 'question': '그중 실내 위주로', 'inherits_region': True})
print(session_notes)
assert session_notes[-1]['inherits_region']

검증 완료: 후속 질문 세션 메모


## 정리

여기서는 최종 앱 전체가 아니라 이 장에서 확인해야 할 핵심 계약만 봤습니다. 같은 원리는 `project_template/app`, `project_template/frontend`, `project_template/data` 안의 실제 파일로 이어집니다.

In [1]:
summary = {
    'notebook': 'completed',
    'next_step': '관련 chapter 문서를 읽고 같은 검증을 테스트로 반복합니다.',
}
print(summary)
assert summary['notebook'] == 'completed'

검증 완료: final summary
